# 03 — Zero-shot baseline: Qwen2.5-1.5B-Instruct

Prompts an instruction-tuned LLM with the full inventory of 60 intents and asks it to name
one. No training, no examples in the prompt — this measures what the model already knows
about Turkish voice commands, and sets the bar that LoRA fine-tuning in notebook 04 has to
clear.

The prompt lives in `src/prompting.py` and is shared with notebook 04, so the only thing
that differs between zero-shot and fine-tuned is the weights.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alper4n/turkish-intent-llm/blob/main/notebooks/03_zeroshot_qwen.ipynb)

## 0. Setup

In [ ]:
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    REPO_URL = "https://github.com/alper4n/turkish-intent-llm.git"
    if not Path("turkish-intent-llm").exists():
        !git clone -q $REPO_URL
    %cd turkish-intent-llm
    !pip install -q -U "transformers>=5.0,<6" "datasets>=3.0" "accelerate>=1.0" "pyyaml>=6.0"

root = Path.cwd()
while not (root / "src").is_dir() and root != root.parent:
    root = root.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
print("repo root:", root)

In [ ]:
import json
import time

import matplotlib.pyplot as plt
import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

from src.data import load_splits, intent_labels
from src.metrics import compute_metrics
from src.prompting import (build_messages, build_system_prompt, generate_labels,
                           parse_label, prepare_tokenizer)
from src.utils import describe_environment, load_config, save_results, set_seed

data_cfg = load_config(root / "configs" / "data.yaml")
cfg = load_config(root / "configs" / "zeroshot_qwen.yaml")
SEED = data_cfg["seed"]
set_seed(SEED)

FIG_DIR = root / data_cfg["paths"]["figures_dir"]
FIG_DIR.mkdir(parents=True, exist_ok=True)

env = describe_environment()
if torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"
env["device"] = DEVICE
print(json.dumps(env, indent=2))

## 1. Data and label inventory

In [ ]:
splits = load_splits(data_cfg["dataset"]["locale"],
                     cache_dir=root / data_cfg["dataset"]["cache_dir"])
LABELS = intent_labels(splits["train"])
print(f"{len(LABELS)} intents, {len(splits['test']):,} test utterances")

## 2. The prompt

One system message carrying all 60 intent names, one user message carrying the utterance.
Decoding is greedy, so the run is reproducible.

In [ ]:
system_prompt = build_system_prompt(LABELS)
print(system_prompt[:400] + "\n...\n")
print("--- example user turn ---")
print(build_messages("yarın sabah yedide alarm kur", LABELS)[1]["content"])

## 3. Load the model

fp16 on every backend: T4 has no bfloat16, and fp16 *inference* is stable on Apple MPS —
unlike fp16 training, which is why notebook 02 stays in fp32 when CUDA is absent.

In [ ]:
MODEL_NAME = cfg["model"]["name"]

tokenizer = prepare_tokenizer(AutoTokenizer.from_pretrained(MODEL_NAME))
start = time.time()
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, dtype=getattr(torch, cfg["model"]["dtype"]))
model.to(DEVICE).eval()

n_params = sum(p.numel() for p in model.parameters())
print(f"loaded in {time.time()-start:.0f}s")
print(f"parameters: {n_params:,}  (trainable in this experiment: 0)")
print(f"prompt length: {len(tokenizer(build_messages('test', LABELS)[0]['content'])['input_ids'])} tokens")

## 4. Check the prompt on validation, not on test

The prompt format is verified on a validation subsample first. Iterating on the test set
would quietly turn this "zero-shot" baseline into a tuning run with an unreported search,
and its number would no longer be comparable with the other two experiments.

In [ ]:
dev_n = cfg["prompt"]["dev_sample_size"]
dev = splits["validation"].sample(dev_n, random_state=SEED).reset_index(drop=True)

dev_raw = generate_labels(
    model, tokenizer, dev["utt"].tolist(), LABELS,
    batch_size=cfg["generation"]["batch_size"],
    max_new_tokens=cfg["generation"]["max_new_tokens"],
    device=DEVICE,
)
dev_pred = [parse_label(text, LABELS) for text in dev_raw]

dev_valid = sum(p in set(LABELS) for p in dev_pred)
dev_acc = sum(p == g for p, g in zip(dev_pred, dev["intent"])) / len(dev)
print(f"\nvalid label outputs: {dev_valid}/{len(dev)} ({dev_valid/len(dev):.1%})")
print(f"validation accuracy (n={len(dev)}): {dev_acc:.4f}")

In [ ]:
print("raw outputs that did not parse to a known intent:")
unparsed = [(raw, parsed) for raw, parsed in zip(dev_raw, dev_pred)
            if parsed not in set(LABELS)]
if unparsed:
    for raw, parsed in unparsed[:10]:
        print(f"  {raw!r} -> {parsed!r}")
else:
    print("  (none — the model always answered with a valid intent name)")

The prompt is now frozen. Everything below runs once on the test set.

## 5. Test set

In [ ]:
test_raw = generate_labels(
    model, tokenizer, splits["test"]["utt"].tolist(), LABELS,
    batch_size=cfg["generation"]["batch_size"],
    max_new_tokens=cfg["generation"]["max_new_tokens"],
    device=DEVICE,
)
y_pred = [parse_label(text, LABELS) for text in test_raw]
y_true = splits["test"]["intent"].tolist()

In [ ]:
scored = compute_metrics(y_true, y_pred, LABELS)

print(f"test utterances      : {scored['n_examples']:,}")
print(f"accuracy             : {scored['accuracy']:.4f}")
print(f"macro-F1 (59 present): {scored['macro_f1']:.4f}")
print(f"macro-F1 (all 60)    : {scored['macro_f1_all_labels']:.4f}")
print(f"invalid predictions  : {scored['label_space']['n_invalid_predictions']}")
for item in scored["label_space"]["invalid_predictions"][:5]:
    print(f"    {item['count']:>4}  {item['prediction']!r}")

### Predictions are saved

A generative model's raw output is data in its own right — the error-analysis notebook needs
the actual strings, not just the scores.

In [ ]:
predictions = pd.DataFrame({
    "id": splits["test"]["id"],
    "utt": splits["test"]["utt"],
    "gold": y_true,
    "raw_output": [text.strip() for text in test_raw],
    "predicted": y_pred,
})
predictions["correct"] = predictions["gold"] == predictions["predicted"]

pred_path = (root / data_cfg["paths"]["results_dir"]
             / f"{cfg['output']['predictions_name']}.csv")
predictions.to_csv(pred_path, index=False)
print("wrote", pred_path)
predictions.head(8)

## 6. Where zero-shot fails

In [ ]:
per_class = pd.DataFrame(scored["per_class"])
print("Worst 12 intents by F1:")
print(per_class.head(12).to_string(index=False))
print()
print(f"intents with F1 = 0: {(per_class['f1'] == 0).sum()} of {len(per_class)}")

In [ ]:
print("Most frequent confusions (gold -> predicted):")
for row in scored["top_confusions"]:
    print(f"  {row['count']:>3}  {row['gold']}  ->  {row['predicted']}")

In [ ]:
# A generative classifier can collapse onto a few "attractor" labels regardless of input.
pred_counts = predictions["predicted"].value_counts()
gold_counts = predictions["gold"].value_counts()
comparison = pd.DataFrame({"predicted": pred_counts, "gold": gold_counts}).fillna(0).astype(int)
comparison["over_prediction"] = comparison["predicted"] - comparison["gold"]
print("Most over-predicted intents (model says it far more often than it occurs):")
print(comparison.nlargest(10, "over_prediction").to_string())
print()
print(f"distinct intents ever predicted: {predictions['predicted'].nunique()} of {len(LABELS)}")

## 7. Against the fine-tuned baseline

In [ ]:
berturk_path = root / data_cfg["paths"]["results_dir"] / "berturk.json"
rows = [{
    "experiment": "Zero-shot Qwen2.5-1.5B",
    "trainable_params": 0,
    "accuracy": scored["accuracy"],
    "macro_f1": scored["macro_f1"],
}]
if berturk_path.exists():
    berturk = json.loads(berturk_path.read_text(encoding="utf-8"))
    rows.append({
        "experiment": "BERTurk fine-tune",
        "trainable_params": berturk["training"]["trainable_parameters"],
        "accuracy": berturk["test"]["accuracy"],
        "macro_f1": berturk["test"]["macro_f1"],
    })
pd.DataFrame(rows).set_index("experiment")

## 8. Save results

In [ ]:
payload = {
    "experiment": cfg["experiment"],
    "model": MODEL_NAME,
    "seed": SEED,
    "config": cfg,
    "environment": env,
    "model_info": {
        "total_parameters": int(n_params),
        "trainable_parameters": 0,
        "dtype": cfg["model"]["dtype"],
    },
    "prompt": {
        "system": system_prompt,
        "dev_sample_size": dev_n,
        "dev_accuracy": round(float(dev_acc), 4),
        "dev_valid_rate": round(dev_valid / len(dev), 4),
    },
    "test": scored,
    "prediction_diversity": {
        "distinct_intents_predicted": int(predictions["predicted"].nunique()),
        "n_intents": len(LABELS),
    },
}

path = save_results(payload, cfg["output"]["results_name"],
                    results_dir=root / data_cfg["paths"]["results_dir"])
print("wrote", path)
print(f"\nHEADLINE  accuracy={scored['accuracy']:.4f}  macro-F1={scored['macro_f1']:.4f}")